<a href="https://colab.research.google.com/github/marcory-hub/yolo11n-on-grove-vision-ai-v2/blob/main/YOLO11_pt_to_vela_2026_02_24.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# int8 to vela for grove vision ai v2

## IMPORTANT: Use runtime 2025.07!

Last accessed: 2026-02-25

- Put dataset and best.pt in root of google drive
- Default: no_post=False

Click [here](https://github.com/HimaxWiseEyePlus/YOLO11_on_WE2?tab=readme-ov-file#open-yolo11_on_we2_tutorialipynb-on-colab) for the himax wise eye plus repository. Currently only support YOLO11 object detection task.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Copy zipped dataset to colab
!cp '/content/drive/MyDrive/dataset.zip' '/content/dataset.zip'

# Unzip quietly to avoid massive logs (remove -q -n flags if needed)
# -q = quiet (no file list output)
# -n = no overwrite (skips existing files)
!unzip -qn '/content/dataset.zip' -d '/content/dataset/'

# Copy model weights
!cp '/content/drive/MyDrive/best.pt' '/content/best.pt'

In [ ]:
# make a calibrationset with seed 42

import os
import random
import shutil

# Configuration
nc = "4"
class_names = "'amel', 'vcra', 'vespsp', 'vvel'"
num_images = 500
original_train_dir = "/content/dataset/train"
original_valid_dir = "/content/dataset/valid"
temp_dir = "/content/temp_subset"

# Set seed for reproducibility
random.seed(42)

# 1. Clean up existing temp folder to ensure a fresh start
if os.path.exists(temp_dir):
    shutil.rmtree(temp_dir)
    print(f"Cleaned up existing directory: {temp_dir}")

def copy_random_subset(src_dir, dst_dir, num_images):
    # Get all valid image files
    img_path = f"{src_dir}/images"
    if not os.path.exists(img_path):
        print(f"Warning: {img_path} not found. Skipping.")
        return 0

    all_images = [f for f in os.listdir(img_path)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    # Randomly sample
    selected = random.sample(all_images, min(num_images, len(all_images)))

    os.makedirs(f"{dst_dir}/images", exist_ok=True)
    os.makedirs(f"{dst_dir}/labels", exist_ok=True)

    count = 0
    for img in selected:
        label = img.rsplit('.', 1)[0] + '.txt'

        # Copy image
        shutil.copy(f"{src_dir}/images/{img}", f"{dst_dir}/images/{img}")

        # Copy label if it exists (null images handled here)
        src_label_path = f"{src_dir}/labels/{label}"
        if os.path.exists(src_label_path):
            shutil.copy(src_label_path, f"{dst_dir}/labels/{label}")
        count += 1
    return count

# Execute copying
train_count = copy_random_subset(original_train_dir, f"{temp_dir}/train", num_images)
valid_count = copy_random_subset(original_valid_dir, f"{temp_dir}/valid", num_images)

# Prepare YAML data
nc_int = int(nc)
names_list = [name.strip().strip("'") for name in class_names.split(',')]

temp_data_string = f"""
train: {temp_dir}/train/images
val: {temp_dir}/valid/images
nc: {nc_int}
names: {names_list}
"""

# Write the new YAML file
with open("/content/temp_data.yaml", 'w') as f:
    f.write(temp_data_string.strip())

print("--- Summary ---")
print(f"Train images sampled: {train_count}")
print(f"Valid images sampled: {valid_count}")
print(f"YAML created at: /content/temp_data.yaml")

# 0.a export original yolo11n int8 tflite with BatchMatMul Operater

In [ ]:
!git clone https://github.com/kris-himax/ultralytics
%cd ultralytics
%pip install .
%cd ..
import ultralytics
ultralytics.checks()

In [ ]:
# Export the YOLO model to INT8 TFLite format
# Default no_post=False, change to True if needed
!yolo export \
    model='best.pt' \
    format='tflite' \
    int8=True \
    no_post=False \
    imgsz=224 \
    data='temp_data.yaml'

# Install vela compiler

In [ ]:
# Install the Arm Ethos-U Vela compiler
!pip3 install ethos-u-vela

# Download the specific Himax configuration for the Grove Vision AI V2
!wget https://raw.githubusercontent.com/HimaxWiseEyePlus/ML_FVP_EVALUATION/main/vela/himax_vela.ini

# Convert yolo11 int8 tflite to vela model

In [ ]:
# Compile the model for the Ethos-U55 NPU (64 MAC configuration)
!vela --accelerator-config ethos-u55-64 \
    --config himax_vela.ini \
    --system-config My_Sys_Cfg \
    --memory-mode My_Mem_Mode_Parent \
    --output-dir ./best_saved_model \
    ./best_saved_model/best_full_integer_quant.tflite

# Check if vela model size

In [ ]:
import os

# Updated path to the specific location of your Vela model
output_model = '/content/best_saved_model/best_full_integer_quant_vela.tflite'

def check_model(path, name):
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"{name} Path: {path}")
        print(f"{name} Size: {size_mb:.2f} MB")
        return size_mb
    else:
        print(f"⚠️ {name} not found at: {path}")
        return None

# Check the size
final_size = check_model(output_model, "Vela Optimized Model")

# Grove Vision AI V2 Constraint Check
if final_size:
    LIMIT = 2.4
    if final_size <= LIMIT:
        print(f"✅ SUCCESS: Model is within the {LIMIT}MB limit.")
    else:
        print(f"❌ ERROR: Model ({final_size:.2f}MB) exceeds the {LIMIT}MB limit!")
        print("Tip: If it's too large, reduce 'imgsz' during export or use a smaller architecture.")

# Compare metrics of best.pt and int8.tflite

In [ ]:
from ultralytics import YOLO
import pandas as pd

# Define paths
DATA_YAML = '/content/dataset/data.yaml'
PT_MODEL_PATH = '/content/best.pt'
TFLITE_MODEL_PATH = '/content/best_saved_model/best_full_integer_quant.tflite'

# 1. Validate PyTorch Model (The Baseline)
print("Evaluating PyTorch Model...")
model_pt = YOLO(PT_MODEL_PATH)
results_pt = model_pt.val(data=DATA_YAML, split='test', imgsz=224, verbose=False)

# 2. Validate TFLite Model (The Quantized Version)
print("Evaluating INT8 TFLite Model...")
model_tflite = YOLO(TFLITE_MODEL_PATH, task='detect')
results_tflite = model_tflite.val(data=DATA_YAML, split='test', imgsz=224, verbose=False, batch=1)

# 3. Create Comparison Table
data = {
    "Metric": ["mAP@50", "mAP@50-95", "Precision", "Recall"],
    "PyTorch (FP32)": [
        results_pt.box.map50,
        results_pt.box.map,
        results_pt.box.mp,
        results_pt.box.mr
    ],
    "TFLite (INT8)": [
        results_tflite.box.map50,
        results_tflite.box.map,
        results_tflite.box.mp,
        results_tflite.box.mr
    ]
}

df = pd.DataFrame(data)
df['Loss (%)'] = ((df['PyTorch (FP32)'] - df['TFLite (INT8)']) / df['PyTorch (FP32)']) * 100
print("\n--- Model Accuracy Comparison ---")
print(df.to_string(index=False))

In [ ]:
import shutil
import os
from ultralytics import YOLO

# 1. Load the TFLite model
model_path = '/content/best_saved_model/best_full_integer_quant.tflite'
model = YOLO(model_path, task='detect')

# 2. Run validation on the ALL images in the 'val' split
# We set 'project' to /content/ and 'name' to 'temp_val' to control the output folder
results = model.val(
    data='/content/dataset/data.yaml',
    split='val',
    imgsz=224,
    plots=True,
    batch=1,
    project='/content',
    name='temp_val_run'
)

# 3. Define source and destination
# YOLO saves results in project/name/ (e.g., /content/temp_val_run/)
source_dir = '/content/temp_val_run'
dest_folder = '/content/best_saved_model'

# 4. Copy the confusion matrix and other evaluation plots to best_saved_model
files_to_copy = ['confusion_matrix.png', 'confusion_matrix_normalized.png', 'results.png', 'PR_curve.png']

for file_name in files_to_copy:
    src_path = os.path.join(source_dir, file_name)
    if os.path.exists(src_path):
        shutil.copy(src_path, os.path.join(dest_folder, file_name))
        print(f"✅ Saved {file_name} to {dest_folder}")

# Clean up temporary folder
shutil.rmtree(source_dir)

# Zip and download the full_integer_quant_vela_imgz_224_nopost.tflite
CHANGE TO POST IF NEEDED

In [ ]:
import shutil
import os
from google.colab import files

# Move best.onnx into the saved_model folder
source_onnx = '/content/best.onnx'
dest_folder = '/content/best_saved_model'

if os.path.exists(source_onnx):
    shutil.move(source_onnx, os.path.join(dest_folder, 'best.onnx'))
    print("Moved best.onnx to best_saved_model folder.")
else:
    print("best.onnx not found at root. It might already be moved.")

# Zip the best_saved_model folder
zip_name = 'full_integer_quant_vela_imgz_224_nopost.tflite'
shutil.make_archive(zip_name, 'zip', dest_folder)
print(f"Created {zip_name}.zip")

# Download the zip file to your computer
files.download(f'{zip_name}.zip')